# Hip Bone Reconstruction — Colab / RunPod Training

End-to-end training notebook. Run cells top-to-bottom on a Colab GPU runtime
(or paste them into a RunPod Jupyter session). The trained checkpoint is
saved to Google Drive (or your RunPod volume) and can be dropped into the
`models/` folder of the web app to enable real reconstructions.

## 1. Environment

In [ ]:
!nvidia-smi || echo 'No GPU detected — switch to a GPU runtime in Colab.'

In [ ]:
# Mount Google Drive on Colab. Skip on RunPod (use the persistent volume instead).
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/hip-recon'
except ImportError:
    WORK = '/workspace/hip-recon'
import os; os.makedirs(WORK, exist_ok=True); print('Workdir:', WORK)

## 2. Clone repo + install

In [ ]:
%cd $WORK
![ -d Claude-hip-try1 ] || git clone https://github.com/bhaskarsdose/claude-hip-try1.git Claude-hip-try1
%cd Claude-hip-try1
!git pull
!pip install -q -r requirements.txt
!pip install -q -e .

## 3. Get hip data

CTPelvic1K is hosted on Zenodo. Drop the relevant Zenodo direct-download
URLs into `urls.txt` (one per line); the loader handles the rest. If you
already have hip masks elsewhere on Drive, set `RAW` to that folder and
skip the download step.

In [ ]:
RAW = 'data/raw/ctpelvic1k'  # nifti masks (.nii.gz) live under here
PROCESSED = 'data/processed'  # 128**3 .npy volumes will be written here
import os; os.makedirs(RAW, exist_ok=True); os.makedirs(PROCESSED, exist_ok=True)
# Edit urls.txt with your Zenodo download URLs, then:
# !python scripts/download_ctpelvic1k.py --urls-from urls.txt --out $RAW
# !find $RAW -name '*.zip' -exec unzip -n {} -d $RAW \;

In [ ]:
# Convert raw masks -> 128**3 .npy training tensors.
# Use --label 2 or --label 3 to keep only one hip; default keeps the whole pelvis.
!python scripts/prepare_dataset.py --in-dir $RAW --out-dir $PROCESSED

## 4. Train

In [ ]:
!python -m hip_recon.train --data-dir $PROCESSED --epochs 100 --batch-size 2 --out models/unet3d_hip.pt

In [ ]:
# Optional: TensorBoard
%load_ext tensorboard
%tensorboard --logdir runs

## 5. Save the checkpoint back to Drive / volume

In [ ]:
import shutil, os
src = 'models/unet3d_hip.pt'
dst = os.path.join(WORK, 'unet3d_hip.pt')
shutil.copy2(src, dst)
print('Saved checkpoint to', dst)
print('Download it locally and drop it into the repo at models/unet3d_hip.pt to enable real inference in the web app.')